# 01 - Simulate increments

Drops new files into the landing volume so the sdp-meta pipelines have something to process.
Pick a batch with the `batch` widget, run all, then run the pipelines (or let the file-arrival job fire).
Each batch prints **what to expect** after the next pipeline run.

| Batch | What lands | Shows |
|---|---|---|
| `1_new_sales` | next N `sales_stream` files from the samples dataset | Auto Loader picks up only new files (feature #8) |
| `2_customer_changes` | `customers_001_changes.csv`: 5 UPDATE, 2 DELETE, 3 INSERT | SCD2 history in silver `customers` (feature #7) |
| `3_bad_data` | bad customers, sales_orders and sales_stream rows | warn / drop / quarantine rules (features #4, #5) |
| `4_fail_rule` | one sales order with a NULL `order_number` | `expect_or_fail` stops **only** the retail_sales pipeline |
| `4_undo_fail` | rewrites the batch 4 file with the record fixed | "fix at source, rerun": the retail_sales pipeline recovers |

Batches 2-4 write fixed file names, so re-running a batch overwrites the same file instead of adding more data.
Use `99_reset` to go back to the baseline.

In [0]:
dbutils.widgets.text("catalog", "mg_aws_serverless_workspace_catalog", "Catalog")
dbutils.widgets.text("prefix", "sdp_meta", "Schema prefix")
dbutils.widgets.dropdown("batch", "1_new_sales",
    ["1_new_sales", "2_customer_changes", "3_bad_data", "4_fail_rule", "4_undo_fail"], "Batch")
dbutils.widgets.text("stream_files", "15", "Batch 1: sales_stream files to add")

CATALOG = dbutils.widgets.get("catalog")
PREFIX = dbutils.widgets.get("prefix")
BATCH = dbutils.widgets.get("batch")
STREAM_FILES = int(dbutils.widgets.get("stream_files"))

SRC = "/Volumes/samples/databricks/datasets/retail-org"
LANDING = f"/Volumes/{CATALOG}/{PREFIX}_landing/files"
REF, SALES = f"{LANDING}/retail_reference", f"{LANDING}/retail_sales"
print(f"batch={BATCH}  landing={LANDING}")

In [0]:
import json

import pandas as pd


def write_json_lines(rows, path):
    """Write a list of dicts as a JSON Lines file (one object per line) to a volume path."""
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print(f"wrote {len(rows)} rows -> {path.replace(LANDING + '/', '')}")


def raw_json_rows(path, n):
    """Return the first n rows of a JSON source as plain dicts, keeping the original field shapes."""
    return [r.asDict(recursive=True) for r in spark.read.json(path).orderBy("order_number").limit(n).collect()]


def customers_baseline():
    """Baseline customers as pandas (all values as strings, same columns/order as the landing CSV)."""
    return spark.read.option("header", True).csv(f"{REF}/customers/customers_000_initial.csv").toPandas()


def current_customers(pdf):
    """Latest version per customer_id, sorted numerically by id so picks are deterministic."""
    pdf = pdf.assign(_id=pdf.customer_id.astype("int64"), _ts=pdf.operation_ts.astype("int64"))
    return pdf.sort_values("_ts").groupby("_id").tail(1).sort_values("_id").drop(columns=["_id", "_ts"])


def write_customers_csv(pdf, name):
    """Write a customers change file with exactly the baseline columns."""
    path = f"{REF}/customers/{name}"
    pdf.to_csv(path, index=False)
    print(f"wrote {len(pdf)} rows -> retail_reference/customers/{name}")

In [0]:
if BATCH == "1_new_sales":
    src_files = sorted(dbutils.fs.ls(f"{SRC}/sales_stream/sales_stream.json"), key=lambda f: f.name)
    landed = {f.name for f in dbutils.fs.ls(f"{SALES}/sales_stream")}
    todo = [f for f in src_files if f.name not in landed][:STREAM_FILES]
    for f in todo:
        dbutils.fs.cp(f.path, f"{SALES}/sales_stream/{f.name}")
    new_rows = sum(spark.read.json(f.path).count() for f in todo)
    print(f"copied {len(todo)} new files ({len(src_files) - len(landed) - len(todo)} still left in samples)")
    print(f"EXPECT: bronze.sales_stream +{new_rows} rows, only these {len(todo)} files read by Auto Loader;"
          f" silver.orders gains the new order_numbers (revisions of an existing order replace it)")

In [0]:
if BATCH == "2_customer_changes":
    base = customers_baseline()
    cur = current_customers(base)
    ts0 = int(base.operation_ts.astype("int64").max()) + 86_400  # one day after the latest baseline change

    upd = cur.iloc[0:5].copy()  # 5 UPDATEs: move up one loyalty segment, buy 10 more units
    upd["loyalty_segment"] = (upd.loyalty_segment.astype(int) + 1).clip(upper=3).astype(str)
    upd["units_purchased"] = (upd.units_purchased.astype(float) + 10).astype(str)
    upd["operation"] = "UPDATE"

    dele = cur.iloc[5:7].copy()  # 2 DELETEs
    dele["operation"] = "DELETE"

    ins = cur.iloc[7:10].copy()  # 3 INSERTs: brand-new customer ids
    next_id = int(base.customer_id.astype("int64").max()) + 1
    ins["customer_id"] = [str(next_id + i) for i in range(3)]
    ins["customer_name"] = [f"DEMO NEW CUSTOMER {i + 1}" for i in range(3)]
    ins["operation"] = "INSERT"

    changes = pd.concat([upd, dele, ins])
    changes["operation_ts"] = [str(ts0 + i) for i in range(len(changes))]
    changes["valid_from"] = changes["operation_ts"]
    changes["valid_to"] = ""
    write_customers_csv(changes[base.columns], "customers_001_changes.csv")
    display(changes[["customer_id", "customer_name", "operation", "loyalty_segment", "units_purchased", "operation_ts"]])
    print("EXPECT: bronze.customers +10 | silver.customers +8 rows (5 new versions + 3 new customers),"
          " 7 rows closed (__END_AT set: 5 updated + 2 deleted) | current customers -2 deleted +3 inserted = +1")

In [0]:
if BATCH == "3_bad_data":
    base = customers_baseline()
    cur = current_customers(base)
    ts0 = int(base.operation_ts.astype("int64").max()) + 2 * 86_400

    # customers: 1 warn-only row (unknown loyalty segment, kept) + 2 quarantined rows
    bad = cur.iloc[10:13].copy()
    bad["operation"] = ["UPDATE", "UPSERT", "UPDATE"]       # row 2: invalid operation
    bad.iloc[0, bad.columns.get_loc("loyalty_segment")] = "9"  # row 1: warn only
    bad.iloc[2, bad.columns.get_loc("customer_id")] = None     # row 3: missing key
    bad["operation_ts"] = [str(ts0 + i) for i in range(3)]
    bad["valid_from"], bad["valid_to"] = bad["operation_ts"], ""
    write_customers_csv(bad[base.columns], "customers_002_bad.csv")

    # sales_orders: 2 new orders with an empty order_datetime -> quarantined
    so = raw_json_rows(f"{SALES}/sales_orders/sales_orders_000_initial.json", 2)
    max_order = spark.read.json(f"{SALES}/sales_orders").agg({"order_number": "max"}).first()[0]
    for i, r in enumerate(so):
        r["order_number"], r["order_datetime"] = max_order + 1 + i, ""
    write_json_lines(so, f"{SALES}/sales_orders/sales_orders_002_bad.json")

    # sales_stream: 1 good + 2 rows without customer_id -> quarantined
    ss = raw_json_rows(f"{SRC}/sales_stream/sales_stream.json", 3)
    max_stream = spark.read.json(f"{SRC}/sales_stream/sales_stream.json").agg({"order_number": "max"}).first()[0]
    for i, r in enumerate(ss):
        r["order_number"] = max_stream + 1000 + i
        if i > 0:
            r["customer_id"] = None
    write_json_lines(ss, f"{SALES}/sales_stream/sales_stream_bad_002.json")

    print("EXPECT: customers bronze +1 / quarantine +2, warn metric known_loyalty_segment failed=1, silver +1 version")
    print("EXPECT: sales_orders bronze +0 / quarantine +2 | sales_stream bronze +1 / quarantine +2 | silver.orders +1")

In [0]:
FAIL_FILE = f"{SALES}/sales_orders/sales_orders_003_fail.json"
FIXED_ORDER_NUMBER = 999_000_001  # order_number used when the bad record is corrected


def fail_row(order_number):
    """The batch 4 order: a copy of the first baseline order with the given order_number."""
    row = raw_json_rows(f"{SALES}/sales_orders/sales_orders_000_initial.json", 1)[0]
    row["order_number"] = order_number
    return row


if BATCH == "4_fail_rule":
    write_json_lines([fail_row(None)], FAIL_FILE)
    print("EXPECT: sdp_meta_bronze_retail_sales update FAILS on expect_or_fail valid_order_number;"
          " retail_reference pipelines are unaffected. Run batch 4_undo_fail, then rerun to recover.")

if BATCH == "4_undo_fail":
    # Fix the record at the source, keeping the SAME file name. Deleting the file does not work:
    # the failed update already logged it as a file to process, so a deleted file raises
    # CloudFileNotFoundException on the retry.
    write_json_lines([fail_row(FIXED_ORDER_NUMBER)], FAIL_FILE)
    print(f"EXPECT: next retail_sales run succeeds; bronze.sales_orders +1 and silver.orders +1"
          f" (order_number {FIXED_ORDER_NUMBER})")

In [0]:
summary = [(f"{grp}/{t}", len(dbutils.fs.ls(f"{LANDING}/{grp}/{t}")))
           for grp, tables in {"retail_reference": ["customers"], "retail_sales": ["sales_orders", "sales_stream"]}.items()
           for t in tables]
display(spark.createDataFrame(summary, "folder STRING, files INT"))